# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [ ]:
# Task 1 — Multi-series line with highlight
asian_df = df[df['Region'] == 'Asia']
HIGHLIGHT = 'China'
GREY = '#DDDDDD'
ACCENT = '#E63946'

fig = go.Figure()

# Grey background lines for context
for country in asian_df['Country'].unique():
    if country == HIGHLIGHT:
        continue
    cdf = asian_df[asian_df['Country'] == country]
    fig.add_trace(go.Scatter(
        x=cdf['Year'], y=cdf['CO2_Mt'],
        mode='lines',
        line=dict(color=GREY, width=1.5),
        showlegend=False,
        hovertemplate=f'{country}: %{{y:,.0f}} Mt<extra></extra>'
    ))

# Highlighted country — China
hdf = asian_df[asian_df['Country'] == HIGHLIGHT]
last = hdf[hdf['Year'] == hdf['Year'].max()].iloc[0]

fig.add_trace(go.Scatter(
    x=hdf['Year'], y=hdf['CO2_Mt'],
    mode='lines',
    line=dict(color=ACCENT, width=3),
    showlegend=False,
    hovertemplate=f'{HIGHLIGHT}: %{{y:,.0f}} Mt<extra></extra>'
))

# Direct label at line end
fig.add_annotation(
    x=last['Year'], y=last['CO2_Mt'],
    text=f'<b>{HIGHLIGHT}</b>',
    xanchor='left', xshift=10, showarrow=False,
    font=dict(family='Arial', size=13, color=ACCENT)
)

fig.update_layout(
    title=dict(
        text="China's CO₂ Output Has Grown 4× Since 2000,<br>Leaving Other Asian Economies Behind",
        font=dict(family='Arial', size=15), x=0
    ),
    xaxis=dict(showgrid=False, showline=False,
               tickfont=dict(family='Arial', size=12)),
    yaxis=dict(title='CO₂ Emissions (Mt)', showgrid=True,
               gridcolor='#F5F5F5', showline=False, zeroline=False,
               tickfont=dict(family='Arial', size=12)),
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(r=120, t=90, l=60, b=50)
)

fig.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [ ]:
# Task 2 — Slopegraph: regional averages 2000 vs 2022
region_agg = (df.groupby(['Region', 'Year'])['CO2_Mt']
              .mean().reset_index())
slope_df = region_agg[region_agg['Year'].isin([2000, 2022])].copy()

pivot = (slope_df.pivot(index='Region', columns='Year', values='CO2_Mt')
         .reset_index())
pivot.columns = ['Region', 'val_2000', 'val_2022']
pivot['increased'] = pivot['val_2022'] > pivot['val_2000']
pivot['color'] = pivot['increased'].map({True: '#E63946', False: '#2A9D8F'})
pivot = pivot.sort_values('val_2022').reset_index(drop=True)

def spread_label_positions(values, min_gap):
    """Push label y-positions apart so no two are closer than min_gap."""
    indexed = sorted(enumerate(values), key=lambda x: x[1])
    positions = [v for _, v in indexed]
    orig_indices = [i for i, _ in indexed]
    for i in range(1, len(positions)):
        if positions[i] - positions[i - 1] < min_gap:
            positions[i] = positions[i - 1] + min_gap
    result = [0.0] * len(values)
    for rank, orig_idx in enumerate(orig_indices):
        result[orig_idx] = positions[rank]
    return result

y_max_data = max(pivot['val_2000'].max(), pivot['val_2022'].max())
min_gap = y_max_data * 0.06   # ~6% of y-range keeps font-11 labels clear

left_pos  = spread_label_positions(pivot['val_2000'].tolist(), min_gap)
right_pos = spread_label_positions(pivot['val_2022'].tolist(), min_gap)
y_ceiling = max(max(left_pos), max(right_pos)) * 1.18

fig = go.Figure()

for idx, (_, row) in enumerate(pivot.iterrows()):
    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[row['val_2000'], row['val_2022']],
        mode='lines+markers',
        line=dict(color=row['color'], width=2.5),
        marker=dict(size=9, color=row['color']),
        showlegend=False,
        hovertemplate=f"{row['Region']}: %{{y:,.0f}} Mt<extra></extra>"
    ))
    # Left label (adjusted y to avoid overlap)
    fig.add_annotation(
        x=2000, y=left_pos[idx],
        text=f"{row['Region']}  {row['val_2000']:,.0f}",
        xanchor='right', xshift=-10, showarrow=False,
        font=dict(family='Arial', size=11, color='#333333')
    )
    # Right label (adjusted y to avoid overlap)
    fig.add_annotation(
        x=2022, y=right_pos[idx],
        text=f"{row['val_2022']:,.0f}  {row['Region']}",
        xanchor='left', xshift=10, showarrow=False,
        font=dict(family='Arial', size=11, color='#333333')
    )

# Colour-key annotations positioned above the data
fig.add_annotation(x=2011, y=y_ceiling * 0.98,
    text='<span style="color:#E63946">■</span> Increased',
    showarrow=False, font=dict(family='Arial', size=12))
fig.add_annotation(x=2011, y=y_ceiling * 0.90,
    text='<span style="color:#2A9D8F">■</span> Decreased',
    showarrow=False, font=dict(family='Arial', size=12))

fig.update_layout(
    title=dict(
        text="Asia's Emissions Surged While Europe and North America<br>Cut CO₂ Over Two Decades",
        font=dict(family='Arial', size=15), x=0
    ),
    xaxis=dict(
        tickvals=[2000, 2022], ticktext=['2000', '2022'],
        showgrid=False, showline=False, range=[1985, 2037],
        tickfont=dict(family='Arial', size=13)
    ),
    yaxis=dict(
        showticklabels=False, showgrid=False,
        showline=False, zeroline=False,
        range=[-100, y_ceiling]
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=200, r=200, t=90, b=50)
)

fig.show()
